# YouTube Video Summarizer

In [24]:
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import pickle
from pydantic import BaseModel
from minsearch import Index
import json

In [2]:
openai_client = OpenAI()

## Data Preprocessing

In [3]:
video_id = 'ph1PxZIkz1o'

!wget https://github.com/alexeygrigorev/ai-bootcamp-codespace/raw/refs/heads/main/week1/ph1PxZIkz1o.bin -q

# ytt_api = YouTubeTranscriptApi()
# transcript = ytt_api.fetch(video_id)

# with open(f'{video_id}.bin', 'wb') as f_out:
#     pickle.dump(transcript, f_out)

In [4]:
with open(f'{video_id}.bin', 'rb') as f_in:
    transcript = pickle.load(f_in)

In [5]:
def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

In [6]:
def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

In [7]:
subtitles = make_subtitles(transcript)

In [8]:
print(subtitles[:500])

0:00 So hi everyone. Uh today we are going to
0:02 talk about our upcoming course. The
0:05 upcoming course is called machine
0:06 learning zoom camp. And um this is
0:10 already I put the link in the
0:12 description. So if you're watching um
0:14 this video in recording or you're
0:17 watching it live, you go here in the
0:19 description after under this video and
0:21 then you see a link course. uh click on
0:25 that link and this bring you will bring
0:27 you to
0:29 this website this GitHub


## Structured Output

In [9]:
class Chapter(BaseModel):
    timestamp: str
    title: str

class YTSummaryResponse(BaseModel):
    summary: str
    chapters: list[Chapter]

In [10]:
instructions = """
    Summarize the transcript and describe the main purpose of the video
    and the main ideas. 

    Also output chapters with time. Use usual sentence case, not Title Case for the chapter.

    More chapters is better than fewer chapters. Have a chapter at least every 3-5 minutes
""".strip()

In [11]:
def llm_structured(instructions, user_prompt, output_format, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_format
    )

    # return response.output[0].content[0].parsed
    return response.output_parsed

In [12]:
summary = llm_structured(
    instructions=instructions,
    user_prompt=subtitles,
    output_format=YTSummaryResponse
)

In [13]:
print(summary.summary)

The video outlines details about an upcoming course called "Machine Learning Zoom Camp," focusing primarily on the course structure, content updates, and prerequisites for potential participants. It emphasizes that the course is free, aimed at teaching essential machine learning engineering skills with a hands-on focus as well as project-based learning. The presenter addresses common questions about the course, including job placement opportunities, prerequisite knowledge, and expectations on project outcomes. The video also highlights that prior programming experience, particularly in Python, is necessary for success in this course.


In [14]:
for chapter in summary.chapters:
    print(chapter.timestamp, chapter.title)

0:00 Introduction and course overview
2:00 Course content structure and updates
5:00 Job placement and skills relevance
8:00 Prerequisites for the course
10:00 Deep learning and computer vision coverage
13:00 Expectations and outcomes for participants
16:00 Recommended resources and companion book
19:00 Course engagement and project requirements
22:00 Clarifying different engineering roles
25:00 Participation in peer reviews
28:00 Q&A session starts
32:00 Project aspects and their importance
35:00 Course environment and setup
39:00 Follow-up resources and next steps
43:00 Final thoughts and conclusion


## Transcript Chunking

In [17]:
def sliding_window(seq, size, step):
    """Create overlapping chunks using sliding window approach."""
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size]
        result.append(batch)
        if i + size >= n:
            break

    return result

In [18]:
def join_lines(transcript) -> str:
    """Join transcript entries into continuous text."""
    lines = []

    for entry in transcript:
        text = entry.text.replace('\n', ' ')
        lines.append(text)

    return ' '.join(lines)

def format_chunk(chunk):
    """Format a chunk with start/end timestamps and text."""
    time_start = format_timestamp(chunk[0].start)
    time_end = format_timestamp(chunk[-1].start)
    text = join_lines(chunk)

    return {
        'start': time_start,
        'end': time_end,
        'text': text
    }

In [ ]:
# split the stranscripts corpus into 60-character chunks with 30 characters of overlap
chunks = []

for chunk in sliding_window(transcript, 60, 30):
    processed = format_chunk(chunk)
    chunks.append(processed)

print(f"Created {len(chunks)} chunks")

Created 46 chunks


In [22]:
for chunk in chunks[:5]:
    print(chunk)

{'start': '0:00', 'end': '2:38', 'text': "So hi everyone. Uh today we are going to talk about our upcoming course. The upcoming course is called machine learning zoom camp. And um this is already I put the link in the description. So if you're watching um this video in recording or you're watching it live, you go here in the description after under this video and then you see a link course. uh click on that link and this bring you will bring you to this website this GitHub page. This GitHub page is the main entry point to our course and um yeah I think it's more or less self-explanatory. If you want to sign up this is the button you click and the actual course starts in on September 15th. it means that it's uh slightly less than one one month before the course starts and the purpose of today's um session is to just answer your questions. So you have some questions and uh you can ask these questions using uh you can ask your questions using the pinned link. So there's a pinned link in t

In [25]:
index = Index(text_fields=["text"])
index.fit(chunks)

In [31]:
def search(query):
    """Search for relevant documents."""
    return index.search(
        query=query,
        num_results=15
    )

In [32]:
instructions = """
    Answer the QUESTION based on the CONTEXT from the subtitles of a YouTube video.

    Use only the facts from the CONTEXT when answering the QUESTION.

    When answering the question, 
    provide the citation in form of the video URL pointing at the timestamp where
    this is discussed. If the question is discussed in multiple documents,
    cite all of them.

    Don't use markdown or any formatting in the output.
""".strip()

prompt_template = """
<VIDEO_ID>
{video_id}
</VIDEO_ID>

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [33]:
def build_prompt(question, search_results):
    context = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=context,
        video_id=video_id
    ).strip()

In [34]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [35]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    response = llm(prompt, instructions=instructions)
    return response

In [36]:
answer = rag('Can I find a job after the course?')
print(answer)

Yes, you can find a job after completing the course; many past participants have successfully found jobs. However, the course does not provide job placements as it is not a boot camp; rather, it teaches essential skills for machine learning engineers. It's recommended to apply the skills learned through projects, possibly starting with volunteering. The skills taught are aligned with what employers look for, which increases your chances of finding a job.

For more information, you can check out the relevant sections from the video at these timestamps: 1:21 (https://www.youtube.com/watch?v=ph1PxZIkz1o&t=81) and 52:34 (https://www.youtube.com/watch?v=ph1PxZIkz1o&t=3154).
